# ZSR Train Network — Graph Analysis
**Pipeline:**
1. Config & imports
2. Phase 1 — Extract station GPS from snapshots (batch)
3. Phase 2 — Build directed graph with delay statistics (batch)
4. Phase 3 — Export: GraphML + Folium HTML map
5. Phase 4 — Diagnostics

## 0. Config & Imports

In [1]:
# ── Paths — edit these to match your layout ───────────────────────────────
PARQUET_PATH      = "../data/train_data.parquet"
STATION_COORDS_PATH = "../data/station_coords.csv"
GRAPHML_PATH      = "../outputs/maps/zsr_graph.graphml"
HTML_MAP_PATH     = "../outputs/maps/zsr_network_map.html"

# Minimum observations on an edge before it's included in the graph
MIN_EDGE_SAMPLES  = 5

In [2]:
import json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import networkx as nx
import folium
from pathlib import Path
from collections import defaultdict

# Ensure output directories exist
Path(GRAPHML_PATH).parent.mkdir(parents=True, exist_ok=True)
Path(HTML_MAP_PATH).parent.mkdir(parents=True, exist_ok=True)

parquet_file = pq.ParquetFile(PARQUET_PATH)
print(f"Parquet row groups: {parquet_file.num_row_groups}")

Parquet row groups: 45


In [3]:
# ── Shared helpers ────────────────────────────────────────────────────────

def load_chunk(parquet_file, i: int) -> pd.DataFrame:
    """Read one row group, parse JSON trains column, explode to one row per train."""
    df_chunk = parquet_file.read_row_group(i).to_pandas()
    if 'trains' not in df_chunk.columns:
        return pd.DataFrame()
    df_chunk['trains'] = df_chunk['trains'].apply(
        lambda x: json.loads(x) if isinstance(x, str) else x
    )
    df = pd.json_normalize(
        df_chunk.explode('trains').reset_index(drop=True)['trains']
    )
    # Common derived columns used in multiple phases
    df['InfoZoStanice'] = df['InfoZoStanice'].astype(str).str.strip()
    df['lat'] = df['Position'].apply(
        lambda p: p[0] if isinstance(p, list) and len(p) == 2 else np.nan
    )
    df['lon'] = df['Position'].apply(
        lambda p: p[1] if isinstance(p, list) and len(p) == 2 else np.nan
    )
    df['Cas_dt']  = pd.to_datetime(df['Cas'],     dayfirst=True, errors='coerce')
    df['plan_dt'] = pd.to_datetime(df['CasPlan'], dayfirst=True, errors='coerce')
    df['date']    = df['Cas_dt'].dt.date
    df['delay_gt'] = (df['Cas_dt'] - df['plan_dt']).dt.total_seconds() / 60
    df['OrderCurrentStation'] = pd.to_numeric(
        df['OrderCurrentStation'], errors='coerce'
    )
    return df


def delay_color(mean_delay: float) -> str:
    """Map a mean delay value to a colour for the Folium map."""
    if mean_delay <= 0: return '#4CAF50'   # green  — on time / early
    if mean_delay <= 2: return '#FFC107'   # yellow — slight
    if mean_delay <= 5: return '#FF9800'   # orange — moderate
    return '#F44336'                        # red    — severe


print('Helpers loaded.')

Helpers loaded.


## 1. Extract Station GPS
Group `(InfoZoStanice, lat, lon)` across all snapshots.  
Uses median GPS per name — robust to the train drifting slightly while the station label is active.

In [5]:
name_lats: dict = defaultdict(list)
name_lons: dict = defaultdict(list)

for i in range(parquet_file.num_row_groups):
    df = load_chunk(parquet_file, i)
    if df.empty:
        continue

    at_station = df[
        df['InfoZoStanice'].notna() &
        (df['InfoZoStanice'] != '') &
        (df['InfoZoStanice'] != 'nan') &
        df['lat'].notna() &
        df['lon'].notna()
    ]

    for _, row in at_station[['InfoZoStanice', 'lat', 'lon']].iterrows():
        name_lats[row['InfoZoStanice']].append(row['lat'])
        name_lons[row['InfoZoStanice']].append(row['lon'])

    print(f'  chunk {i+1:>2}/{parquet_file.num_row_groups} '
          f'— {len(name_lats)} station names with GPS so far')

print(f'\nDone. {len(name_lats)} unique station names found.')

/tmp/ipykernel_1288543/2071644243.py:22: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['Cas_dt']  = pd.to_datetime(df['Cas'],     dayfirst=True, errors='coerce')
/tmp/ipykernel_1288543/2071644243.py:23: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['plan_dt'] = pd.to_datetime(df['CasPlan'], dayfirst=True, errors='coerce')


  chunk  1/45 — 378 station names with GPS so far


/tmp/ipykernel_1288543/2071644243.py:22: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['Cas_dt']  = pd.to_datetime(df['Cas'],     dayfirst=True, errors='coerce')
/tmp/ipykernel_1288543/2071644243.py:23: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['plan_dt'] = pd.to_datetime(df['CasPlan'], dayfirst=True, errors='coerce')


  chunk  2/45 — 378 station names with GPS so far


/tmp/ipykernel_1288543/2071644243.py:22: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['Cas_dt']  = pd.to_datetime(df['Cas'],     dayfirst=True, errors='coerce')
/tmp/ipykernel_1288543/2071644243.py:23: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['plan_dt'] = pd.to_datetime(df['CasPlan'], dayfirst=True, errors='coerce')


...

  chunk 43/45 — 416 station names with GPS so far


/tmp/ipykernel_1288543/2071644243.py:22: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['Cas_dt']  = pd.to_datetime(df['Cas'],     dayfirst=True, errors='coerce')
/tmp/ipykernel_1288543/2071644243.py:23: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['plan_dt'] = pd.to_datetime(df['CasPlan'], dayfirst=True, errors='coerce')


  chunk 44/45 — 416 station names with GPS so far


/tmp/ipykernel_1288543/2071644243.py:22: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['Cas_dt']  = pd.to_datetime(df['Cas'],     dayfirst=True, errors='coerce')
/tmp/ipykernel_1288543/2071644243.py:23: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['plan_dt'] = pd.to_datetime(df['CasPlan'], dayfirst=True, errors='coerce')


  chunk 45/45 — 416 station names with GPS so far

Done. 416 unique station names found.


In [6]:
# Build coords DataFrame and save
records = []
for name in name_lats:
    lats, lons = name_lats[name], name_lons[name]
    records.append({
        'name':       name,
        'lat':        round(float(np.median(lats)), 6),
        'lon':        round(float(np.median(lons)), 6),
        'n_samples':  len(lats),
        # IQR spread: > 0.01° (~1 km) suggests GPS jitter or name collision
        'lat_spread': round(float(np.percentile(lats, 75) - np.percentile(lats, 25)), 4),
        'lon_spread': round(float(np.percentile(lons, 75) - np.percentile(lons, 25)), 4),
    })

coords_df = pd.DataFrame(records).sort_values('n_samples', ascending=False)
coords_df.to_csv(STATION_COORDS_PATH, index=False)

print(f'Total station names mapped: {len(coords_df)}')
print(f'High GPS spread (lat IQR > 0.01°): {(coords_df["lat_spread"] > 0.01).sum()}')
print()
noisy = coords_df[coords_df['lat_spread'] > 0.01].sort_values('lat_spread', ascending=False)
if noisy.empty:
    print('No stations with unreliable GPS ✓')
else:
    print('=== Stations with unreliable GPS ===')
    print(noisy[['name', 'lat', 'lon', 'lat_spread', 'n_samples']].to_string(index=False))
print()
print('=== Top 20 by snapshot count ===')
print(coords_df.head(20)[['name', 'lat', 'lon', 'n_samples']].to_string(index=False))

Total station names mapped: 416
High GPS spread (lat IQR > 0.01°): 0

No stations with unreliable GPS ✓

=== Top 20 by snapshot count ===
                     name       lat       lon  n_samples
Bratislava hlavná stanica 48.158900 17.106700     111835
                   Košice 48.722910 21.268720      79994
                   Trnava 48.369779 17.584820      62025
                    Čadca 49.445280 18.786810      55761
                   Žilina 49.227100 18.746040      50936
                   Púchov 49.113770 18.327170      48482
               Nové Zámky 47.995090 18.174130      42119
                     Kúty 48.661950 17.047530      41633
           Starý Smokovec 49.139298 20.222847      41523
                   Vrútky 49.115190 18.924420      41455
                    Kysak 48.852880 21.223930      40746
                  Galanta 48.185000 17.722970      40146
            Štrbské Pleso 49.118530 20.064100      38525
                   Prešov 48.983379 21.249126      37449
       

## 2. Build Directed Graph with Delay Statistics
For each `(CisloVlaku, date)` run: sort by `OrderCurrentStation`, deduplicate,  
then create directed edges between consecutive `InfoZoStanice` values.  
Edge attributes = delay statistics aggregated across all observations.

In [7]:
# Accumulators
edge_delays:      dict = defaultdict(list)   # (from, to) → [delay_gt, ...]
edge_train_types: dict = defaultdict(set)    # (from, to) → {TypVlaku, ...}

for i in range(parquet_file.num_row_groups):
    df = load_chunk(parquet_file, i)
    if df.empty:
        continue

    df = df[
        df['InfoZoStanice'].notna() &
        (df['InfoZoStanice'] != '') &
        (df['InfoZoStanice'] != 'nan')
    ]

    for (train, date), grp in df.groupby(['CisloVlaku', 'date']):
        g = (
            grp.sort_values('OrderCurrentStation')
            .drop_duplicates('OrderCurrentStation')
            .reset_index(drop=True)
        )
        if len(g) < 2:
            continue

        for idx in range(len(g) - 1):
            from_name  = g.iloc[idx]['InfoZoStanice']
            to_name    = g.iloc[idx + 1]['InfoZoStanice']
            delay      = g.iloc[idx + 1]['delay_gt']
            train_type = g.iloc[idx].get('TypVlaku', '?')

            if from_name == to_name:
                continue

            edge_delays[(from_name, to_name)].append(delay)
            edge_train_types[(from_name, to_name)].add(str(train_type))

    print(f'  chunk {i+1:>2}/{parquet_file.num_row_groups} '
          f'— {len(edge_delays)} unique sections found')

print(f'\nDone. {len(edge_delays)} candidate edges before filtering.')

/tmp/ipykernel_1288543/2071644243.py:22: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['Cas_dt']  = pd.to_datetime(df['Cas'],     dayfirst=True, errors='coerce')
/tmp/ipykernel_1288543/2071644243.py:23: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['plan_dt'] = pd.to_datetime(df['CasPlan'], dayfirst=True, errors='coerce')


  chunk  1/45 — 851 unique sections found


/tmp/ipykernel_1288543/2071644243.py:22: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['Cas_dt']  = pd.to_datetime(df['Cas'],     dayfirst=True, errors='coerce')
/tmp/ipykernel_1288543/2071644243.py:23: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['plan_dt'] = pd.to_datetime(df['CasPlan'], dayfirst=True, errors='coerce')


  chunk  2/45 — 942 unique sections found


/tmp/ipykernel_1288543/2071644243.py:22: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['Cas_dt']  = pd.to_datetime(df['Cas'],     dayfirst=True, errors='coerce')
/tmp/ipykernel_1288543/2071644243.py:23: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['plan_dt'] = pd.to_datetime(df['CasPlan'], dayfirst=True, errors='coerce')


...

  chunk 43/45 — 1667 unique sections found


/tmp/ipykernel_1288543/2071644243.py:22: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['Cas_dt']  = pd.to_datetime(df['Cas'],     dayfirst=True, errors='coerce')
/tmp/ipykernel_1288543/2071644243.py:23: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['plan_dt'] = pd.to_datetime(df['CasPlan'], dayfirst=True, errors='coerce')


  chunk 44/45 — 1673 unique sections found


/tmp/ipykernel_1288543/2071644243.py:22: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['Cas_dt']  = pd.to_datetime(df['Cas'],     dayfirst=True, errors='coerce')
/tmp/ipykernel_1288543/2071644243.py:23: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['plan_dt'] = pd.to_datetime(df['CasPlan'], dayfirst=True, errors='coerce')


  chunk 45/45 — 1707 unique sections found

Done. 1707 candidate edges before filtering.


In [8]:
# Build NetworkX DiGraph from accumulators
name_to_coords = {
    row['name']: (row['lat'], row['lon'])
    for _, row in coords_df.iterrows()
    if pd.notna(row['lat'])
}

G = nx.DiGraph()

for (from_name, to_name), delays in edge_delays.items():
    delays_clean = [d for d in delays if pd.notna(d) and -30 <= d <= 180]
    if len(delays_clean) < MIN_EDGE_SAMPLES:
        continue

    arr = np.array(delays_clean)

    for name in (from_name, to_name):
        if name not in G:
            coords = name_to_coords.get(name, (None, None))
            G.add_node(name, lat=coords[0], lon=coords[1])

    G.add_edge(
        from_name, to_name,
        delay_mean   = round(float(np.mean(arr)),             2),
        delay_median = round(float(np.median(arr)),           2),
        delay_p90    = round(float(np.percentile(arr, 90)),   2),
        delay_p95    = round(float(np.percentile(arr, 95)),   2),
        delay_std    = round(float(np.std(arr)),              2),
        delay_skew   = round(float(np.mean(arr) - np.median(arr)), 2),
        sample_count = len(delays_clean),
        train_types  = ', '.join(sorted(edge_train_types[(from_name, to_name)])),
    )

print(f'Graph built: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

# Check missing coords
no_coords = [
    n for n, d in G.nodes(data=True)
    if d.get('lat') is None or (isinstance(d.get('lat'), float) and np.isnan(d['lat']))
]
print(f'Nodes without GPS coordinates: {len(no_coords)}')
if no_coords:
    print('  (these will be skipped in the map but kept in the graph)')
    print('  Sample:', no_coords[:10])

Graph built: 385 nodes, 1018 edges
Nodes without GPS coordinates: 0


## 3. Export

In [9]:
# GraphML — for Gephi and future graph feature lookups
nx.write_graphml(G, GRAPHML_PATH)
print(f'GraphML saved → {GRAPHML_PATH}')

GraphML saved → ../outputs/maps/zsr_graph.graphml


In [10]:
# Folium interactive HTML map
m = folium.Map(location=[48.7, 19.5], zoom_start=8, tiles='CartoDB dark_matter')

# ── Edges ─────────────────────────────────────────────────────────────────
for from_name, to_name, data in G.edges(data=True):
    from_lat = G.nodes[from_name].get('lat')
    from_lon = G.nodes[from_name].get('lon')
    to_lat   = G.nodes[to_name].get('lat')
    to_lon   = G.nodes[to_name].get('lon')

    # Skip edges where either endpoint has no GPS
    if any(v is None for v in (from_lat, from_lon, to_lat, to_lon)):
        continue
    if any(np.isnan(v) for v in (from_lat, from_lon, to_lat, to_lon)):
        continue

    tooltip = (
        f"<b>{from_name} → {to_name}</b><br>"
        f"Mean delay:   {data['delay_mean']} min<br>"
        f"Median delay: {data['delay_median']} min<br>"
        f"P90 delay:    {data['delay_p90']} min<br>"
        f"P95 delay:    {data['delay_p95']} min<br>"
        f"Std dev:      {data['delay_std']} min<br>"
        f"Skew:         {data['delay_skew']} min<br>"
        f"Samples:      {data['sample_count']}<br>"
        f"Train types:  {data['train_types']}"
    )
    weight = min(1 + np.log1p(data['sample_count']) / 3, 6)

    folium.PolyLine(
        locations=[(from_lat, from_lon), (to_lat, to_lon)],
        color=delay_color(data['delay_mean']),
        weight=weight,
        opacity=0.8,
        tooltip=folium.Tooltip(tooltip, sticky=True),
    ).add_to(m)

# ── Nodes ─────────────────────────────────────────────────────────────────
for name, data in G.nodes(data=True):
    lat = data.get('lat')
    lon = data.get('lon')
    if lat is None or lon is None:
        continue
    if np.isnan(lat) or np.isnan(lon):
        continue

    degree      = G.degree(name)
    in_delays   = [G.edges[u, name]['delay_mean'] for u, _ in G.in_edges(name)]
    avg_in_delay = float(np.mean(in_delays)) if in_delays else 0.0

    tooltip = (
        f"<b>{name}</b><br>"
        f"Connections: {degree}<br>"
        f"Avg incoming delay: {avg_in_delay:.1f} min<br>"
        f"Junction: {'Yes' if degree >= 4 else 'No'}"
    )

    folium.CircleMarker(
        location=[lat, lon],
        radius=4 + min(degree, 10),
        color='white',
        fill=True,
        fill_color=delay_color(avg_in_delay),
        fill_opacity=0.9,
        tooltip=folium.Tooltip(tooltip, sticky=True),
    ).add_to(m)

    if degree >= 4:
        folium.Marker(
            location=[lat, lon],
            icon=folium.DivIcon(
                html=f'<div style="font-size:9px;color:white;white-space:nowrap;">{name}</div>',
                icon_size=(150, 20),
                icon_anchor=(0, 0),
            )
        ).add_to(m)

# ── Legend ─────────────────────────────────────────────────────────────────
m.get_root().html.add_child(folium.Element("""
<div style="position:fixed;bottom:30px;left:30px;z-index:1000;
            background:rgba(0,0,0,0.8);color:white;padding:12px;
            border-radius:8px;font-size:13px;">
  <b>Mean delay on section</b><br>
  <span style="color:#4CAF50">●</span> On time / early (≤ 0 min)<br>
  <span style="color:#FFC107">●</span> Slight (0–2 min)<br>
  <span style="color:#FF9800">●</span> Moderate (2–5 min)<br>
  <span style="color:#F44336">●</span> Severe (&gt; 5 min)<br>
  <br><b>Line thickness</b> = traffic volume
</div>
"""))

m.save(HTML_MAP_PATH)
print(f'Map saved → {HTML_MAP_PATH}')

Map saved → ../outputs/maps/zsr_network_map.html


## 4. Diagnostics

In [11]:
components  = list(nx.weakly_connected_components(G))
top_nodes   = sorted(G.degree(), key=lambda x: x[1], reverse=True)[:10]
top_edges   = sorted(G.edges(data=True), key=lambda x: x[2]['delay_mean'], reverse=True)[:10]

print(f'Nodes:                       {G.number_of_nodes()}')
print(f'Edges:                       {G.number_of_edges()}')
print(f'Weakly connected components: {len(components)}')
print(f'Largest component:           {max(len(c) for c in components)} nodes')
print(f'Isolated nodes:              {len(list(nx.isolates(G)))}')
print()

print('Top 10 hubs (by degree):')
for name, deg in top_nodes:
    print(f'  {name:<40} degree={deg}')
print()

print('Top 10 highest delay sections (mean, min samples=5):')
for u, v, d in top_edges:
    print(f'  {u} → {v}')
    print(f'    mean={d["delay_mean"]} min  '
          f'p90={d["delay_p90"]} min  '
          f'samples={d["sample_count"]}')

# Nodes still missing coordinates after GPS extraction
no_coords = [
    n for n, d in G.nodes(data=True)
    if d.get('lat') is None or
    (isinstance(d.get('lat'), float) and np.isnan(d['lat']))
]
print(f'\nNodes without GPS: {len(no_coords)}')
if no_coords:
    print('  These appear in sequences but were never observed as InfoZoStanice.')
    print('  They are included in graph topology but hidden from the map.')
    for n in sorted(no_coords)[:20]:
        print(f'    {n}')

Nodes:                       385
Edges:                       1018
Weakly connected components: 2
Largest component:           376 nodes
Isolated nodes:              0

Top 10 hubs (by degree):
  Bratislava hlavná stanica                degree=17
  Odb. Vinohrady                           degree=13
  Košice                                   degree=12
  Nové Mesto nad Váhom                     degree=12
  Bratislava-Nové Mesto                    degree=12
  Šurany                                   degree=12
  Moldava nad Bodvou                       degree=12
  Kysak                                    degree=11
  Trnava                                   degree=11
  Pezinok                                  degree=11

Top 10 highest delay sections (mean, min samples=5):
  Topoľčany → Chynorany
    mean=32.2 min  p90=38.8 min  samples=5
  Gbelce → Štúrovo tranzitná skupina
    mean=20.6 min  p90=40.5 min  samples=10
  Čierna nad Tisou → Čierna nad Tisou št. hr.
    mean=18.94 min  p90=47.0